## K-Nearest Neighbour Construction

### Import Libraries

In [2]:
import pandas as pd
import numpy as np
import networkx as nx

# PyTorch and PyTorch Geometric
import torch

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression

# Pyvis
from pyvis.network import Network


### Performing Log Regression to Feature Select 

In [4]:
# Load dataset
dataset = pd.read_csv('../data/processed/processed_dataset.csv')

# drop irrelevant columns
df_log = dataset.drop(columns=['imei_list', 'phone_no_m']) 
df_log.drop(columns=[ "voc_hour_mode",'voc_day_mode','sms_hour_mode','sms_day_mode', 'voc_dayname_mode_count', 'sms_dayname_mode_count', 'flow_sum', 'call_county_unique', 'voc_hour_mode_count', 'sms_day_mode_count'], inplace=True)

# Fill NaN values with the median of each column
df_log = df_log.fillna(0)

X = df_log.drop(columns=["label"]) 
y = df_log["label"]

# 🔹 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4211, stratify=y)

# 🔹 Standardize Features (Logistic Regression is sensitive to feature scaling)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# # 🔹 Train Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

feature_names = X.columns  # Make sure X is a DataFrame

# Convert X_train and X_test into DataFrames with column names
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)


# Get model coefficients
coef = model.coef_[0]

# Create coefficient DataFrame
coeff_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coef
})

# Filter by abs(coef) > 0.05
coeff_df['AbsCoefficient'] = np.abs(coeff_df['Coefficient'])
important_features_df = coeff_df[coeff_df['AbsCoefficient'] > 0.25]

# Get list of selected features
selected_features = important_features_df['Feature'].tolist()
columns_to_keep = selected_features

# drop irrelevant columns
df = dataset.drop(columns=['imei_list']) 
df= df.fillna(0)
columns_to_keep += ["phone_no_m","label"]
df = df[columns_to_keep]

#Check the result
print(df['label'].value_counts())
print(df.shape)


# Separate the labels and features
labels = df[['phone_no_m', 'label']]  # Extract label columns
features = df.drop(columns=['label'])  # Drop the label column to keep only the features


# Scale the features (exclude 'phone_no_m' from scaling)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features.drop(columns=['phone_no_m']))  # Scale the features

# Store the scaled features and labels
feature_dict = pd.DataFrame(scaled_features, index=features["phone_no_m"]).to_dict(orient="index")  # Store features as a dictionary
label_dict = labels.set_index("phone_no_m").to_dict(orient="index")  # Store labels as a dictionary

# Compute cosine similarity on the scaled features (instead of PCA features)
similarity_matrix = cosine_similarity(scaled_features)  # Cosine similarity directly on scaled features
similarity_df = pd.DataFrame(similarity_matrix, index=df["phone_no_m"], columns=df["phone_no_m"])  # Create a similarity DataFrame

print("similarity_df shape:", similarity_df.shape)


label
0    4144
1    1962
Name: count, dtype: int64
(6106, 28)
similarity_df shape: (6106, 6106)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Creation of Cosine Similarity Matrix 

In [5]:
# Create Graph
G = nx.Graph()

for idx, row in df.iterrows():
    G.add_node(row['phone_no_m'], label=row['label'])


k_nearest_neighbours = 15  ## during testing, this value has been adjusted from 10,15,20,50

for i, phone1 in enumerate(df["phone_no_m"]):
    
    similarities = [
        (j, similarity_matrix[i, j])
        for j in range(len(df["phone_no_m"])) if i != j
    ]

    positive = sorted([p for p in similarities], key=lambda x: x[1], reverse=True)

    # Select neighbors based strictly on k-nearest neighbours
    selected = []
    selected += positive[:k_nearest_neighbours]

    # Step 6: Add edges to graph (skip padding!)
    for neighbor_idx, sim in selected:
        neighbor = df["phone_no_m"].iloc[neighbor_idx]
        G.add_edge(phone1, neighbor, weight=sim)


# Map node IDs to indices
node_mapping = {node: idx for idx, node in enumerate(G.nodes())}

# Convert edges into PyTorch Geometric format
edge_index = torch.tensor([(node_mapping[u], node_mapping[v]) for u, v in G.edges()], dtype=torch.long).t()

# Convert edge weights
edge_attr = torch.tensor([G[u][v]['weight'] for u, v in G.edges()], dtype=torch.float)

print("Graph Construction Done! Number of Nodes:", len(G.nodes()), "Number of Edges:", len(G.edges()))



Graph Construction Done! Number of Nodes: 6106 Number of Edges: 65570


### Creating an HTML to Plot a Visualization of the Graph

In [6]:
# creating a html to plot a visualisation of the graph, can be skipped if visualisation is not needed
net = Network(notebook=True, height="750px", width="100%", bgcolor="white", font_color="black", directed=True)

# Select a subset of nodes 
num_nodes_to_display = 600  # 10% of actual graph, if not computation will be too slow
nodes_to_display = list(G.nodes)[:num_nodes_to_display]

# Add nodes with their attributes and color coding (only for selected nodes)
for node in nodes_to_display:
    node_color = "#808080"  # Default color (gray) if label is missing

    # Assign color based on the label (0 = blue, 1 = red)
    if node in label_dict:
        label_value = label_dict[node]['label']
        node_color = "salmon" if label_value == 0 else "skyblue"

    net.add_node(node, color=node_color, label = " ")

# Add edges for high-similarity pairs (with cosine similarity as weight)
for i, phone1 in enumerate(df["phone_no_m"]):
    for j, phone2 in enumerate(df["phone_no_m"]):
        # Only add edges that connect the selected nodes
        if phone1 in nodes_to_display and phone2 in nodes_to_display and i != j:
            if G.has_edge(phone1, phone2):
                similarity_score = similarity_matrix[i, j]
                edge_tooltip = f"Similarity: {similarity_score:.2f}"
                net.add_edge(phone1, phone2, value=similarity_score, title=edge_tooltip)

# Save visualization as HTML
net.save_graph("subgraph_network_graph.html")